In [1]:
import sys
print(sys.executable)


/Users/cauasturaro/Documents/CODE/learning/intent-bench/.venv/bin/python


In [2]:
from huggingface_hub import HfApi

info = HfApi().dataset_info("PolyAI/banking77")
print(info.sha)


1846984189f2bc0ac51c6fc3a600fa9e1714e0c6


In [3]:
from datasets import load_dataset

DATASET_ID = "PolyAI/banking77"
REVISION = "1846984189f2bc0ac51c6fc3a600fa9e1714e0c6"

ds = load_dataset(DATASET_ID, revision=REVISION)
ds


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

In [4]:
print(ds["train"].features)

{'text': Value('string'), 'label': ClassLabel(names=['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical

In [20]:
# How many
label_feature = ds["train"].features["label"]
label_names = label_feature.names
print(len(label_names))

77


In [26]:
# First 5
print(label_names[:5])

['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up']


In [22]:
# Does testing and training have the same classes?
print(ds["train"].features["label"] == ds["test"].features["label"])

True


In [33]:
# Converting example
example = ds["train"][0]
print(example)
print(label_feature.int2str(example["label"]))

{'text': 'I am still waiting on my card?', 'label': 11}
card_arrival


In [39]:
# Converting To Pandas 
train = ds["train"].to_pandas()
test = ds["test"].to_pandas()
train["intent"] = train["label"].map(lambda i: label_names[i])
test["intent"] = test["label"].map(lambda i: label_names[i])


In [46]:
train_counts = train["intent"].value_counts()
test_counts = test["intent"].value_counts()

In [47]:
# train
train_counts.head(), train_counts.tail(), train_counts.max() / train_counts.min()

(intent
 card_payment_fee_charged                            187
 direct_debit_payment_not_recognised                 182
 balance_not_updated_after_cheque_or_cash_deposit    181
 wrong_amount_of_cash_received                       180
 cash_withdrawal_charge                              177
 Name: count, dtype: int64,
 intent
 lost_or_stolen_card         82
 card_swallowed              61
 card_acceptance             59
 virtual_card_not_working    41
 contactless_not_working     35
 Name: count, dtype: int64,
 np.float64(5.3428571428571425))

In [49]:
# Test
test_counts.head(), test_counts.tail(), test_counts.max() / test_counts.min()

(intent
 card_arrival                        40
 card_linking                        40
 exchange_rate                       40
 card_payment_wrong_exchange_rate    40
 extra_charge_on_statement           40
 Name: count, dtype: int64,
 intent
 cash_withdrawal_charge     40
 card_about_to_expire       40
 apple_pay_or_google_pay    40
 verify_my_identity         40
 country_support            40
 Name: count, dtype: int64,
 np.float64(1.0))

In [51]:
# Text size
train["n_words"] = train["text"].str.split().str.len()
train["n_words"].describe(percentiles=[.5, .95, .99])


count    10003.000000
mean        11.949415
std          7.891577
min          2.000000
50%         10.000000
95%         29.000000
99%         43.000000
max         79.000000
Name: n_words, dtype: float64

In [54]:
def normalize(text: str) -> str:
    return " ".join(text.lower().split())
    ...
train["norm"] = train["text"].map(normalize)
test["norm"] = test["text"].map(normalize)


In [59]:
train["norm"].duplicated(keep=False).sum()                 # coubt duplicates

np.int64(8)

In [60]:
(train.groupby("norm")["label"].nunique() > 1).sum()       # same text with differnt labels

np.int64(0)

In [61]:
test["norm"].isin(set(train["norm"])).sum()      # overlap train/test

np.int64(7)